# 奇偶深度校验 (Odd-Even Transit Check)

**目的**：对 Person1 的候选列表 `master_candidates.csv` 逐个目标做奇偶深度校验，识别食双星(EB)假信号。

**物理原理**：食双星常由亮度不同的两颗星组成，主食(主星被掩)与次食(伴星被掩)深度不同；按周期折叠后，交替的凌日即主食/次食轮流出现。真行星每次凌日深度一致，EB 则奇/偶次深度有系统差异。

**流程**：
```
① 下载光变曲线      lightkurve 按 Target + Sector 取 TESS 数据(可拼接多扇区)
② 预处理           remove_nans → normalize → flatten(压平基线到 ~1.0)
③ 定位凌日历元 t0   CSV 无 t0：在给定 Period 附近跑窄网格 BLS 求 t0/duration
④ 分奇偶测深度      astropy compute_stats 直接给出 depth_odd / depth_even 及误差
⑤ 算显著性 σ        σ = |d_odd − d_even| / √(e_odd² + e_even²)
⑥ 判定             σ > 3 → 不通过(疑双星)；否则通过
```

**输出**：在原表基础上新增 `odd_even_sigma`、`n_transits`、`odd_even_reason`、`odd_even_pass` 四列，保存为 `odd_even_result.csv`（不改动 Person1 原始列）。

## 0. 安装依赖

In [ ]:
!pip install -q numpy pandas astropy lightkurve

## 1. 配置与加载数据

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from collections import Counter

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# ---------------- 参数 ----------------
ODD_EVEN_SIGMA_THRESHOLD = 3.0     # σ > 3 判定不通过(疑双星)
STITCH_ALL_SECTORS = True          # 拼接目标全部扇区 -> 更多凌日，奇偶校验更稳
TESS_CACHE_DIR = 'tess_cache'      # 光变曲线下载缓存目录
os.makedirs(TESS_CACHE_DIR, exist_ok=True)

# ---------------- 输入/输出 ----------------
_PATHS = ['master_candidates.csv', 'attachments/master_candidates.csv']
INPUT_CSV = next((p for p in _PATHS if os.path.exists(p)), _PATHS[0])
OUTPUT_CSV = 'odd_even_result.csv'

ORIG_DF = pd.read_csv(INPUT_CSV)   # 原始数据：只读
df = ORIG_DF.copy()                # 工作副本：仅追加新列
ORIGINAL_COLUMNS = list(ORIG_DF.columns)
print(f'已加载 {INPUT_CSV} -> {len(df)} 个候选')
df.head()

## 2. 核心函数

`compute_odd_even_sigma` 对每个目标返回 `sigma`、可读的失败原因 `reason` 与有效凌日次数 `n_transits`，方便定位「无法判定」卡在哪一步。

In [ ]:
import lightkurve as lk
from astropy.timeseries import BoxLeastSquares

_LC_CACHE = {}

def get_lightcurve(target, sector):
    '''返回 (lightcurve, note)。note ∈ {ok, no_search_result, download_failed}。'''
    key = (str(target), int(sector))
    if key in _LC_CACHE:
        return _LC_CACHE[key]
    search = None
    # 优先指定 sector 的 SPOC，逐级退化到任意来源
    for kw in ({'sector': int(sector), 'author': 'SPOC'},
               {'author': 'SPOC'},
               {'sector': int(sector)},
               {}):
        try:
            sr = lk.search_lightcurve(str(target), **kw)
            if sr is not None and len(sr) > 0:
                search = sr
                break
        except Exception:
            continue
    if search is None:
        _LC_CACHE[key] = (None, 'no_search_result')
        return _LC_CACHE[key]
    lc = None
    try:
        if STITCH_ALL_SECTORS and len(search) > 1:
            lc = search.download_all(download_dir=TESS_CACHE_DIR).stitch()   # 多扇区拼接
        else:
            lc = search[0].download(download_dir=TESS_CACHE_DIR)
    except Exception:
        try:
            lc = search[0].download(download_dir=TESS_CACHE_DIR)
        except Exception:
            lc = None
    note = 'ok' if lc is not None else 'download_failed'
    _LC_CACHE[key] = (lc, note)
    return _LC_CACHE[key]

def compute_odd_even_sigma(target, sector, period, duration_hr):
    '''返回 dict: {sigma, reason, n_transits, depth_odd, depth_even}。reason=='ok' 时 sigma 有效。'''
    out = {'sigma': np.nan, 'reason': '', 'n_transits': 0,
           'depth_odd': np.nan, 'depth_even': np.nan}
    lc, note = get_lightcurve(target, sector)
    if lc is None:
        out['reason'] = note
        return out
    try:
        # 预处理：去 NaN、归一化、压平基线(window 需远大于凌日时长以免削掉凌日)
        lc = lc.remove_nans().normalize().flatten(window_length=401)
        t = np.asarray(lc.time.value, dtype=float)
        f = np.asarray(lc.flux.value, dtype=float)
        good = np.isfinite(t) & np.isfinite(f)
        t, f = t[good], f[good]
        if len(t) < 200:
            out['reason'] = f'few_cadences({len(t)})'
            return out
        # 噪声估计
        mad = np.nanmedian(np.abs(f - np.nanmedian(f)))
        scale = 1.4826 * mad if mad > 0 else np.nanstd(f)
        dy = np.full(len(t), scale if scale > 0 else 1e-3)
        # 在已知周期附近细化，求 t0 与 duration
        bls = BoxLeastSquares(t, f, dy=dy)
        p0 = float(period)
        periods = np.linspace(p0 * 0.999, p0 * 1.001, 300)
        dur0 = max(float(duration_hr), 0.5) / 24.0
        durations = np.unique(np.clip(np.array([dur0 * 0.6, dur0, dur0 * 1.4]),
                                      0.5 / 24.0, 8.0 / 24.0))
        res = bls.power(periods, durations, objective='snr')
        j = int(np.argmax(res.power))
        # 奇偶深度：astropy 直接给出 depth_odd / depth_even 各(值, 误差)
        stats = bls.compute_stats(float(res.period[j]),
                                  float(res.duration[j]),
                                  float(res.transit_time[j]))
        n_tr = int(np.sum(np.atleast_1d(stats['per_transit_count']) > 0))
        out['n_transits'] = n_tr
        d_odd,  e_odd  = float(stats['depth_odd'][0]),  float(stats['depth_odd'][1])
        d_even, e_even = float(stats['depth_even'][0]), float(stats['depth_even'][1])
        out['depth_odd'], out['depth_even'] = d_odd * 1e6, d_even * 1e6
        comb = np.sqrt(e_odd ** 2 + e_even ** 2)
        if not np.isfinite(comb) or comb <= 0:
            out['reason'] = f'oddeven_err_undefined(n_transits={n_tr})'
            return out
        out['sigma'] = abs(d_odd - d_even) / comb
        out['reason'] = 'ok'
        return out
    except Exception as e:
        out['reason'] = f'exception:{type(e).__name__}:{str(e)[:60]}'
        out['n_transits'] = -1
        return out

## 2.1 单目标连通性诊断（先跑这一格）

用又深又短周期、凌日多的 **TOI-1516**（P=2.06d, 深度 12867ppm, SNR 209）验证「下载+BLS」链路。若这一格都失败，说明是**网络/MAST 访问**问题，而非数据本身。

In [ ]:
print('lightkurve 版本:', lk.__version__)
_dbg = ('TOI-1516', 57, 2.0559, 2.45)
try:
    _sr = lk.search_lightcurve(_dbg[0], author='SPOC')
    print('search_lightcurve 命中结果数:', 0 if _sr is None else len(_sr))
except Exception as e:
    print('search_lightcurve 直接报错:', repr(e))
_res = compute_odd_even_sigma(*_dbg)
print('诊断结果:', _res)

r = _res['reason']
if r == 'no_search_result':
    print('>> 结论：搜不到数据。MAST 不可达 / 无网络 / 需代理。批量全部无法判定即源于此。')
elif r == 'download_failed':
    print('>> 结论：能搜到但下载失败。多为网络中断或缓存目录权限；重试或更换 TESS_CACHE_DIR。')
elif r.startswith('few_cadences'):
    print('>> 结论：下载成功但有效点太少(数据质量问题)。')
elif r.startswith('oddeven'):
    print('>> 结论：下载与 BLS 正常，仅该目标凌日不足以拆分奇偶(数据限制)。')
elif r.startswith('exception'):
    print('>> 结论：代码/依赖异常，按异常类型排查(常见为 lightkurve 版本 API 差异)。')
elif r == 'ok':
    print('>> 结论：全链路正常！可继续批量。')

## 3. 批量奇偶校验

逐目标运行，实时打印 σ、有效凌日次数与原因；末尾给出「无法判定」的原因分布。

In [ ]:
sigmas, reasons, ntr_list, dodd_list, deven_list = [], [], [], [], []
for i, row in df.iterrows():
    r = compute_odd_even_sigma(row['Target'], row['Sector'],
                               row['Period (d)'], row['Duration (hr)'])
    sigmas.append(r['sigma']); reasons.append(r['reason'])
    ntr_list.append(r['n_transits'])
    dodd_list.append(r['depth_odd']); deven_list.append(r['depth_even'])
    tag = 'NaN' if not np.isfinite(r['sigma']) else f"{r['sigma']:.2f}"
    print(f"[{i+1:>2}/{len(df)}] {str(row['Target']):<12} sigma={tag:<6} "
          f"n_transits={r['n_transits']:<3} reason={r['reason']}")

df['depth_odd_ppm']  = dodd_list
df['depth_even_ppm'] = deven_list
df['n_transits']     = ntr_list
df['odd_even_sigma'] = sigmas
df['odd_even_reason'] = reasons
df['odd_even_pass'] = pd.array(
    [(x <= ODD_EVEN_SIGMA_THRESHOLD) if np.isfinite(x) else pd.NA for x in sigmas],
    dtype='boolean')

print('\n奇偶校验完成 ->  通过:', int((df['odd_even_pass'] == True).sum()),
      ' 不通过:', int((df['odd_even_pass'] == False).sum()),
      ' 无法判定:', int(df['odd_even_pass'].isna().sum()))
print('无法判定原因分布:')
for reason, cnt in Counter([rs for rs, s in zip(reasons, sigmas)
                            if not np.isfinite(s)]).most_common():
    print(f'   {cnt:>3}  x  {reason}')

## 4. 保存结果

In [ ]:
# 一致性检查：原始数据未被改动
assert df[ORIGINAL_COLUMNS].equals(ORIG_DF[ORIGINAL_COLUMNS]), '原始数据被意外修改!'

NEW_COLS = ['depth_odd_ppm', 'depth_even_ppm', 'n_transits',
            'odd_even_sigma', 'odd_even_reason', 'odd_even_pass']
result = df[ORIGINAL_COLUMNS + NEW_COLS].copy()
result.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print('已保存 ->', OUTPUT_CSV, ' (原始数据完整性校验通过)')

# 展示被判定为疑似双星(不通过)的目标
failed = result[result['odd_even_pass'] == False]
print(f'\n判定不通过(σ>{ODD_EVEN_SIGMA_THRESHOLD}, 疑似双星)的目标: {len(failed)} 个')
result

## 5. 说明

- `odd_even_pass`：`True`=奇偶一致(通过)，`False`=σ>3 疑似双星，`<NA>`=无法判定。
- 若大量 `<NA>`，看批量末尾**原因分布**：`no_search_result/download_failed`=网络问题；`oddeven_err_undefined`=单扇区凌日太少(数据限制，已开启多扇区拼接缓解)。
- 保守原则：无法判定的目标记为缺失，不当作通过。
- 全流程只新增列、不改动 Person1 原始字段。